# PySpark Window Functions — Part 7
## Aggregate Windows | Ranking Windows | Value Windows
### `row_number()`, `rank()`, `dense_rank()`, `lag()`, `lead()`

> **Learning goal:** SQL window functions are already familiar, so this chapter focuses on how to express them with PySpark's DataFrame API, how to choose the correct window frame, and how they support real Data Engineering workflows such as deduplication, customer analytics, event sequencing, anomaly detection, and incremental change analysis.
>
> **Data theme:** E-commerce orders and event-processing records. The sample dataset is readable; a Spark-generated dataset with one million rows is included for scale experiments.

---

# Part 1 — Window functions: concept before code

## 1. What is a window function?

A **window function** calculates a result for a row by looking at a defined set of related rows, called a *window*, while keeping the original row in the output.

An aggregation such as `groupBy("country").sum("amount")` usually collapses many input rows into one row per country. A window aggregation such as `sum("amount").over(Window.partitionBy("country"))` adds a country total to **each original row** in that country.

### Grouped aggregation versus window aggregation

**Grouped aggregation:**
```python
orders_df.groupBy("country").agg(
    sum("amount").alias("country_amount")
)
```
Output grain: one row per country.

**Window aggregation:**
```python
w_country = Window.partitionBy("country")
orders_df.withColumn(
    "country_amount",
    sum("amount").over(w_country)
)
```
Output grain: still one row per order, with the country total attached to each order.

This difference is one of the most important ideas in the topic.

## 2. What is a window specification?

In PySpark, the `Window` class defines which rows are related to the current row.

```python
from pyspark.sql.window import Window

w = Window.partitionBy("country").orderBy("event_date")
```

A window specification can define three things:

1. **Partition (`partitionBy`)** — the logical group within which calculations happen, such as one customer or one country. This is not the same thing as permanently repartitioning the DataFrame.
2. **Order (`orderBy`)** — the order of rows within each window, required for sequence-sensitive operations such as ranking, `lag`, and `lead`.
3. **Frame (`rowsBetween` or `rangeBetween`)** — the subset of rows or ordering values considered relative to the current row for applicable aggregate/value calculations.

Example:
```python
w_customer_time = (
    Window
    .partitionBy("customer_id")
    .orderBy("event_ts", "order_id")
)
```
This says: independently for each customer, arrange records by event timestamp and then by order ID to make the sequence deterministic when timestamps tie.

## 3. The three main families

| Family | Common functions | Typical question |
|---|---|---|
| **Aggregate window functions** | `sum`, `avg`, `count`, `min`, `max` with `.over(...)` | What is the running total, customer total, or rolling average while preserving every row? |
| **Ranking window functions** | `row_number`, `rank`, `dense_rank`, `ntile`, `percent_rank`, `cume_dist` | Which record is first/latest? What is the rank within a category? |
| **Value / analytic window functions** | `lag`, `lead`, `first_value`, `last_value`, `nth_value` | What was the previous event? What happens next? What is the first/last value in a defined frame? |

This chapter gives extensive practice with the requested functions and uses aggregate/ranking/value examples in production-style patterns.

## 4. `partitionBy()` versus `orderBy()`

```python
Window.partitionBy("customer_id")
```
Creates a logical window per customer. Every row for a customer belongs to the same partition of the window calculation.

```python
Window.orderBy("event_ts")
```
Orders rows for a sequence-sensitive calculation. If there is no `partitionBy`, the window is global: all rows are in one logical window. On a large dataset, a global order can be expensive.

```python
Window.partitionBy("customer_id").orderBy("event_ts")
```
Orders events independently for each customer.

**Important:** `partitionBy()` in the window specification defines calculation groups. It does not mean the data has been physically repartitioned in storage. Spark may shuffle data to execute the window.

## 5. Why a deterministic sort matters

If two orders for the same customer have the same timestamp, then this may be ambiguous:

```python
Window.partitionBy("customer_id").orderBy("event_ts")
```

Prefer a tie-breaker:
```python
Window.partitionBy("customer_id").orderBy("event_ts", "order_id")
```

For `row_number`, `lag`, and `lead`, tie-breaking changes which record is considered first/previous/next. In production, choose a stable, meaningful tie-breaker such as event ID, source sequence, or ingestion timestamp.

## 6. Window frames: `rowsBetween()` and `rangeBetween()`

A frame controls which ordered rows are included for functions that use a frame.

```python
w_running = (
    Window.partitionBy("customer_id")
    .orderBy("event_ts", "order_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
```

- `Window.unboundedPreceding`: start at the first row in the partition.
- `Window.currentRow`: end at the current row.
- The frame means “from the beginning of this customer's ordered history through the current row,” which creates a running calculation.

A rolling three-row frame can be written:
```python
.rowsBetween(-2, Window.currentRow)
```
It includes the current row and up to two preceding rows.

`rowsBetween` uses **row positions**. `rangeBetween` uses values of the ordering expression and has different semantics. Do not swap them casually. `rangeBetween` can be especially subtle when there are duplicate order values; read the frame definition carefully.

## 7. Core window functions at a glance

| Function | Meaning | Important parameter |
|---|---|---|
| `row_number()` | Unique sequential number in each ordered window | Window order |
| `rank()` | Rank with gaps after ties | Window order |
| `dense_rank()` | Rank without gaps after ties | Window order |
| `lag(col, offset, default)` | Value from a preceding row | Column, offset, default |
| `lead(col, offset, default)` | Value from a following row | Column, offset, default |
| `sum(col).over(w)` | Total within the window/frame | Aggregate expression and window |
| `avg(col).over(w)` | Mean within the window/frame | Aggregate expression and window |
| `count(col).over(w)` | Count non-null values within the window/frame | Aggregate expression and window |
| `min/max(col).over(w)` | Extreme value within the window/frame | Aggregate expression and window |

---

# Part 2 — Build a realistic order dataset

## 1. Create a sample DataFrame

Run this setup once. The later examples reuse `orders_df`.

```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when, concat, lpad, to_timestamp, to_date,
    row_number, rank, dense_rank, lag, lead,
    sum, avg, count, min, max, first, last, nth_value,
    round, datediff, abs
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType
)

spark = SparkSession.builder.appName("PySparkWindowFunctions").getOrCreate()

data = [
    (1001, "C001", "India", "Electronics", "Delivered", 2499.0, 2, "2026-10-01 09:00:00"),
    (1002, "C001", "India", "Books",        "Delivered",  450.0, 3, "2026-10-01 11:00:00"),
    (1003, "C001", "India", "Home",         "Returned",  1800.0, 1, "2026-10-03 10:00:00"),
    (1004, "C002", "India", "Electronics",  "Delivered", 5600.0, 2, "2026-10-02 13:00:00"),
    (1005, "C002", "India", "Fashion",      "Pending",    950.0, 2, "2026-10-04 08:30:00"),
    (1006, "C002", "India", "Books",        "Delivered",  550.0, 2, "2026-10-04 09:00:00"),
    (1007, "C003", "UK",    "Electronics",  "Delivered",  899.0, 1, "2026-10-02 10:00:00"),
    (1008, "C003", "UK",    "Books",        "Delivered",  650.0, 4, "2026-10-06 15:00:00"),
    (1009, "C003", "UK",    "Fashion",      "Delivered", 1300.0, 3, "2026-10-10 12:00:00"),
    (1010, "C004", "Singapore", "Electronics", "Delivered", 3200.0, 1, "2026-10-03 12:00:00"),
    (1011, "C004", "Singapore", "Home",      "Pending",   1700.0, 1, "2026-10-09 14:00:00"),
    (1012, "C005", "Singapore", "Electronics", "Delivered", 4100.0, 2, "2026-10-07 09:30:00"),
    (1013, "C005", "Singapore", "Fashion",   "Returned",   700.0, 1, "2026-10-08 10:00:00"),
    (1014, "C006", "UK",    "Home",         "Cancelled",  800.0, 1, "2026-10-08 16:00:00"),
    (1015, "C006", "UK",    "Books",        "Delivered",  720.0, 2, "2026-10-11 10:00:00"),
    (1016, "C007", "India", "Electronics",  "Delivered", 7200.0, 3, "2026-10-10 18:00:00"),
    (1017, "C007", "India", "Home",         "Delivered", 1500.0, 1, "2026-10-12 08:00:00"),
    (1018, "C008", "UK",    "Fashion",      "Delivered", 1300.0, 3, "2026-10-12 11:00:00"),
    (1019, "C008", "UK",    "Home",         "Pending",   2200.0, 1, "2026-10-13 09:00:00"),
    (1020, "C009", "India", "Books",        "Delivered",  300.0, 2, "2026-10-13 10:00:00")
]

schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", StringType(), True),
    StructField("country", StringType(), True),
    StructField("category", StringType(), True),
    StructField("status", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("event_ts_text", StringType(), True)
])

orders_df = (
    spark.createDataFrame(data, schema)
    .withColumn("event_ts", to_timestamp("event_ts_text", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("event_date", to_date("event_ts"))
    .drop("event_ts_text")
)

orders_df.orderBy("customer_id", "event_ts", "order_id").show(truncate=False)
```

### Why these columns matter

- `customer_id` lets us calculate customer-level histories.
- `event_ts` orders events precisely; `event_date` supports daily summaries.
- `order_id` is a stable tie-breaker for this exercise.
- `amount` and `quantity` support running totals and averages.
- `category`, `country`, and `status` support ranking and operational metrics.

This is a compact sample with varied order histories. It is designed for learning; it is not a statistically representative dataset.

## 2. Create a one-million-row dataset using Spark

Use this to experiment with window execution on larger data. `spark.range()` and column expressions generate records in Spark rather than building a million Python tuples on the driver.

```python
N = 1_000_000

large_orders_df = spark.range(0, N).select(
    (col("id") + 1_000_000).cast("long").alias("order_id"),
    concat(lit("C"), lpad((col("id") % 100_000).cast("string"), 6, "0")).alias("customer_id"),
    when((col("id") % 3) == 0, "India")
        .when((col("id") % 3) == 1, "UK")
        .otherwise("Singapore").alias("country"),
    when((col("id") % 4) == 0, "Electronics")
        .when((col("id") % 4) == 1, "Books")
        .when((col("id") % 4) == 2, "Home")
        .otherwise("Fashion").alias("category"),
    when((col("id") % 5) == 0, "Returned")
        .when((col("id") % 5) == 1, "Pending")
        .otherwise("Delivered").alias("status"),
    ((col("id") % 10_000) + 100).cast("double").alias("amount"),
    ((col("id") % 5) + 1).cast("int").alias("quantity"),
    to_timestamp(
        concat(
            date_add(to_date(lit("2026-01-01")), (col("id") % 365).cast("int")).cast("string"),
            lit(" 12:00:00")
        )
    ).alias("event_ts")
)

print("Rows:", large_orders_df.count())
```

The values are deterministic and patterned, so this is a scale/performance test, not real customer behaviour. Start with one million rows and increase only if the Spark environment has enough memory and compute.

---

# Part 3 — Aggregate window functions

## Definition

An **aggregate window function** applies an aggregate such as `sum`, `avg`, `count`, `min`, or `max` over a window. Unlike `groupBy`, it can preserve every input row and attach the aggregate result to each row.

**General syntax**
```python
df.withColumn(
    "metric_name",
    sum("amount").over(window_spec)
)
```

Parameters:
- `sum("amount")`: the aggregation expression.
- `.over(window_spec)`: tells Spark which rows belong to the window/frame.
- `withColumn("metric_name", ...)`: adds the computed value as a column without intentionally changing the number of rows.

## Example 1 — Customer lifetime amount within the dataset

```python
w_customer = Window.partitionBy("customer_id")

result = orders_df.withColumn(
    "customer_amount_total",
    sum("amount").over(w_customer)
)
```

`partitionBy("customer_id")` defines one window per customer. `sum("amount")` totals that customer's amount values. `.over(w_customer)` applies the total to each row belonging to that customer. Each order remains in the output, so the same customer total repeats across that customer's rows.

**Real use:** attach customer-level features to transaction rows. This is a total within the available dataset, not necessarily true lifetime spend.

## Example 2 — Running amount per customer

```python
w_running = (
    Window.partitionBy("customer_id")
    .orderBy("event_ts", "order_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

result = orders_df.withColumn(
    "running_amount",
    sum("amount").over(w_running)
)
```

`partitionBy` resets the running total for each customer. `orderBy` defines the sequence. `rowsBetween(unboundedPreceding, currentRow)` includes all preceding rows in that customer's sequence and the current row. This produces a cumulative sum.

## Example 3 — Running quantity per customer

```python
result = orders_df.withColumn(
    "running_units",
    sum("quantity").over(w_running)
)
```

The window definition is reused, but the aggregated column changes from amount to quantity. The cumulative result measures units ordered so far in the available customer history. It is not a count of orders.

## Example 4 — Rolling average of the current and previous two orders

```python
w_last_3 = (
    Window.partitionBy("customer_id")
    .orderBy("event_ts", "order_id")
    .rowsBetween(-2, Window.currentRow)
)

result = orders_df.withColumn(
    "rolling_avg_amount_3_orders",
    avg("amount").over(w_last_3)
)
```

`rowsBetween(-2, currentRow)` selects up to three row positions: two preceding rows plus the current row. For a customer's first row, fewer than three rows exist, so the frame contains only the available rows. `avg` ignores null amounts.

**Use case:** smooth short-term customer spending patterns.

## Example 5 — Count events per customer while keeping each event

```python
result = orders_df.withColumn(
    "customer_order_count",
    count("*").over(w_customer)
)
```

`count("*")` counts all rows in each customer's window. Unlike grouped `count`, the count appears on each order row. This is useful when downstream logic needs each event plus a customer-level count.

## Example 6 — Earliest and latest order timestamps per customer

```python
result = orders_df.withColumn(
    "first_order_ts", min("event_ts").over(w_customer)
).withColumn(
    "latest_order_ts", max("event_ts").over(w_customer)
)
```

`min` finds the earliest timestamp and `max` the latest timestamp in each customer's window. The original order rows remain. These fields can be used for customer tenure features or recency calculations.

## Example 7 — Percentage of customer amount contributed by each order

```python
result = orders_df.withColumn(
    "customer_amount_total", sum("amount").over(w_customer)
).withColumn(
    "share_of_customer_amount",
    col("amount") / col("customer_amount_total")
)
```

The first column calculates a customer total. The second divides each order's amount by that total. This is a row-level share. For production, guard against null or zero denominators and define whether returned/cancelled orders should contribute.

## Example 8 — Running total by country across event order

```python
w_country_running = (
    Window.partitionBy("country")
    .orderBy("event_ts", "order_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

result = orders_df.withColumn(
    "country_running_amount",
    sum("amount").over(w_country_running)
)
```

The running total resets by country rather than customer. All rows in a country are sequenced by timestamp and order ID. **Use case:** a cumulative country-level amount timeline. The value depends on the chosen order and included statuses.

## Example 9 — Rolling average on the large dataset

```python
w_large_rolling = (
    Window.partitionBy("customer_id")
    .orderBy("event_ts", "order_id")
    .rowsBetween(-2, Window.currentRow)
)

large_rolling_df = large_orders_df.withColumn(
    "rolling_avg_amount",
    avg("amount").over(w_large_rolling)
)
```

This applies a rolling three-row average to one million records. It can be expensive because Spark must arrange rows within each customer window. For experimentation, inspect the physical plan and Spark UI. Avoid displaying or collecting the full result.

## Example 10 — Create a customer feature table without losing transaction grain

```python
customer_features_on_orders = orders_df.withColumn(
    "customer_amount_total", sum("amount").over(w_customer)
).withColumn(
    "customer_avg_amount", avg("amount").over(w_customer)
).withColumn(
    "customer_order_count", count("*").over(w_customer)
)
```

This adds three customer-level metrics to every transaction. It can be useful for scoring each order relative to customer history, but the totals include all rows present in this DataFrame. If a model must only use information available before the current order, use a running frame ending at the current row or a frame ending at the previous row as appropriate to avoid future-data leakage.

---

# Part 4 — Ranking window functions

## Definition

Ranking functions assign a position or rank to each row within an ordered window. They are often used for top-N selection, latest-record selection, category leaderboards, deduplication, and percentile-like grouping.

## Rank tie behaviour

Suppose scores are `100, 90, 90, 80`:

| score | `row_number()` | `rank()` | `dense_rank()` |
|---:|---:|---:|---:|
| 100 | 1 | 1 | 1 |
| 90 | 2 | 2 | 2 |
| 90 | 3 | 2 | 2 |
| 80 | 4 | 4 | 3 |

- `row_number()` always assigns a unique sequence number; ties are still assigned separate row numbers.
- `rank()` gives tied rows the same rank and leaves a gap after a tie.
- `dense_rank()` gives tied rows the same rank but does not leave a gap.

Use a deterministic tie-breaker with `row_number()` when exactly one row must win. Do not add a unique tie-breaker to a `rank()` ordering if you intend equal scores to share a rank.

---

# Part 5 — `row_number()`: definition and 10 examples

## Definition and parameters

`row_number()` assigns sequential integers starting at 1 to rows in each ordered window.

**Syntax**
```python
row_number().over(Window.partitionBy("key").orderBy("sort_column"))
```

- `row_number()` takes no column argument.
- `.over(window_spec)` defines the partition and ordering.
- The ordering controls which row receives 1, 2, 3, and so on.
- If ordering values tie, add a stable tie-breaker for deterministic results.

### Example 1 — Number each customer's orders

```python
w_customer_order = Window.partitionBy("customer_id").orderBy("event_ts", "order_id")

result = orders_df.withColumn(
    "customer_order_sequence",
    row_number().over(w_customer_order)
)
```

Each customer's earliest order receives 1, next receives 2, etc. `partitionBy` resets numbering per customer. `orderBy` sorts the sequence. `order_id` breaks timestamp ties. **Use case:** customer event sequence.

### Example 2 — Select the latest order per customer

```python
w_latest = Window.partitionBy("customer_id").orderBy(
    col("event_ts").desc(), col("order_id").desc()
)

latest_orders = orders_df.withColumn(
    "rn", row_number().over(w_latest)
).filter(
    col("rn") == 1
).drop("rn")
```

Descending timestamp places the latest event first, so row number 1 identifies the latest record. Filtering keeps one row per customer. This is a common deduplication/latest-state pattern. Confirm whether order ID is an appropriate tie-breaker.

### Example 3 — Select the first order per customer

```python
w_first = Window.partitionBy("customer_id").orderBy("event_ts", "order_id")

first_orders = orders_df.withColumn(
    "rn", row_number().over(w_first)
).filter(col("rn") == 1)
```

Ascending timestamp puts the earliest order first. Filtering for `rn == 1` keeps one record per customer. Use this for first-touch analysis or first observed order, not necessarily the customer's true first-ever order if the dataset is incomplete.

### Example 4 — Top three orders by amount per country

```python
w_top_amount = Window.partitionBy("country").orderBy(
    col("amount").desc(), col("order_id")
)

top_three = orders_df.withColumn(
    "rn", row_number().over(w_top_amount)
).filter(col("rn") <= 3)
```

The window resets per country. Descending amount sorts highest to lowest. `rn <= 3` keeps at most three rows per country. Because `row_number` breaks ties into distinct positions, this returns at most three rows, not all rows tied at the third amount.

### Example 5 — Deduplicate by source business key

```python
w_dedup = Window.partitionBy("customer_id", "event_ts").orderBy(
    col("order_id").desc()
)

deduplicated = orders_df.withColumn(
    "rn", row_number().over(w_dedup)
).filter(col("rn") == 1).drop("rn")
```

Rows with the same customer and timestamp form a group; the largest order ID wins. This is illustrative: in real data, use the true duplicate key and a meaningful quality/recency rule. Do not arbitrarily discard records when they may be legitimate events.

### Example 6 — Rank records within each category by amount, keeping a unique sequence

```python
w_category = Window.partitionBy("category").orderBy(
    col("amount").desc(), col("order_id")
)

result = orders_df.withColumn("category_position", row_number().over(w_category))
```

Each category gets its own sequential numbering. A unique order ID breaks ties, so every row gets a distinct position. This is suitable when a dashboard requires an exact ordering.

### Example 7 — First delivered order per customer

```python
delivered = orders_df.filter(col("status") == "Delivered")

first_delivered = delivered.withColumn(
    "rn", row_number().over(w_first)
).filter(col("rn") == 1)
```

The filter is applied before the window, so numbering considers delivered orders only. The earliest delivered order per customer receives 1. The output cannot show customers who have no delivered orders because they were removed by the filter.

### Example 8 — Latest event per customer and category

```python
w_customer_category_latest = Window.partitionBy(
    "customer_id", "category"
).orderBy(col("event_ts").desc(), col("order_id").desc())

latest_by_category = orders_df.withColumn(
    "rn", row_number().over(w_customer_category_latest)
).filter(col("rn") == 1)
```

The partition key is a combination of customer and category. One latest record is retained for each observed combination. This can support latest category interaction features or current-state extraction.

### Example 9 — Add row numbers to one million rows per customer

```python
w_large_sequence = Window.partitionBy("customer_id").orderBy("event_ts", "order_id")

large_numbered_df = large_orders_df.withColumn(
    "customer_row_number", row_number().over(w_large_sequence)
)
```

This demonstrates the operation at scale. It can require sorting within each customer group. Inspect Spark UI task durations and shuffle. Do not call `collect()` on the full numbered result.

### Example 10 — Paginate within a partition or pick a segment

```python
ranked = orders_df.withColumn(
    "rn", row_number().over(w_category)
)

positions_4_to_6 = ranked.filter((col("rn") >= 4) & (col("rn") <= 6))
```

This keeps positions 4–6 within each category, not global DataFrame positions. The condition uses `&` to combine Spark column predicates. **Use case:** retrieve a specific segment from a per-category ordered list.

---

# Part 6 — `rank()`: definition and 10 examples

## Definition and parameters

`rank()` assigns the same rank to tied ordering values and leaves gaps after ties. If two rows tie at rank 2, the next rank is 4.

**Syntax**
```python
rank().over(Window.partitionBy("group").orderBy(col("score").desc()))
```

- `rank()` takes no column argument.
- The ordering columns determine what counts as a tie.
- If you add a unique ID to the ordering, otherwise equal scores no longer tie.
- Use descending order for highest-value-first leaderboards.

### Example 1 — Rank orders by amount within country

```python
w_country_amount = Window.partitionBy("country").orderBy(col("amount").desc())

result = orders_df.withColumn("amount_rank", rank().over(w_country_amount))
```

Each country has its own leaderboard. Equal amounts receive the same rank; a rank gap follows ties. No unique tie-breaker is included because equal amounts should remain tied.

### Example 2 — Rank products/categories by amount

```python
w_category_amount = Window.partitionBy("category").orderBy(col("amount").desc())

result = orders_df.withColumn("amount_rank_in_category", rank().over(w_category_amount))
```

This ranks order records within each category by amount. It is not a product-level rank unless the input is already aggregated to product grain. The input grain matters.

### Example 3 — Keep top three rank positions, including ties

```python
ranked = orders_df.withColumn("r", rank().over(w_country_amount))
top_ranked = ranked.filter(col("r") <= 3)
```

Rows tied at rank 3 are all retained. Since ranks can have gaps, this may return more than three rows per country. Use `row_number` if exactly three rows are required.

### Example 4 — Rank by amount within each status

```python
w_status_amount = Window.partitionBy("status").orderBy(col("amount").desc())

result = orders_df.withColumn("status_amount_rank", rank().over(w_status_amount))
```

Each status has its own rank sequence. This can compare high-value records within a lifecycle state. Be careful about treating a high cancelled-order amount as recognized revenue.

### Example 5 — Rank customer totals

```python
customer_totals = orders_df.groupBy("customer_id").agg(
    sum("amount").alias("customer_total")
)

w_customer_total_rank = Window.orderBy(col("customer_total").desc())

ranked_customers = customer_totals.withColumn(
    "customer_total_rank", rank().over(w_customer_total_rank)
)
```

First, grouped aggregation produces one row per customer. Then a global window ranks those customer totals. This global ordering may be expensive at high scale because all customer totals must be ordered. Use only if a global leaderboard is required.

### Example 6 — Rank within country by customer total

```python
customer_country_totals = orders_df.groupBy("country", "customer_id").agg(
    sum("amount").alias("customer_total")
)

w_country_customer_rank = Window.partitionBy("country").orderBy(
    col("customer_total").desc()
)

ranked = customer_country_totals.withColumn(
    "rank_in_country", rank().over(w_country_customer_rank)
)
```

The aggregation changes the grain to country-customer; the window ranks customers independently within each country. This is more meaningful than ranking raw order rows when the question concerns customer totals.

### Example 7 — Rank by quantity within category

```python
w_quantity = Window.partitionBy("category").orderBy(col("quantity").desc())

result = orders_df.withColumn("quantity_rank", rank().over(w_quantity))
```

Rows with equal quantities tie. The rank describes order quantity, not amount. Choose the metric that matches the business question.

### Example 8 — Rank by most recent event

```python
w_recency_rank = Window.partitionBy("customer_id").orderBy(col("event_ts").desc())

result = orders_df.withColumn("recency_rank", rank().over(w_recency_rank))
```

Latest timestamps get rank 1. If two records share the same timestamp, they both receive rank 1. This can be useful when all simultaneous latest events should be retained.

### Example 9 — Rank on the large dataset

```python
w_large_rank = Window.partitionBy("country").orderBy(col("amount").desc())

large_ranked_df = large_orders_df.withColumn(
    "country_amount_rank", rank().over(w_large_rank)
)
```

The window orders one million records within each country. This can be a costly operation because each country contains many rows. If only top 10 per country are needed, still filter by rank after calculating it; inspect the plan and resource usage.

### Example 10 — Compare ranking with row number and dense rank

```python
w_demo = Window.partitionBy("country").orderBy(col("amount").desc())

comparison = orders_df.withColumn("row_num", row_number().over(w_demo)) \
    .withColumn("rank_value", rank().over(w_demo)) \
    .withColumn("dense_rank_value", dense_rank().over(w_demo))
```

This shows the three behaviours side by side. If there are no ties, the rank values may match. When ties exist, `row_number` assigns separate numbers, `rank` leaves gaps, and `dense_rank` does not.

---

# Part 7 — `dense_rank()`: definition and 10 examples

## Definition and parameters

`dense_rank()` assigns the same rank to tied values but does not leave gaps after ties. If scores are `100, 90, 90, 80`, dense ranks are `1, 2, 2, 3`.

### Example 1 — Dense-rank amounts within country

```python
w_country_amount = Window.partitionBy("country").orderBy(col("amount").desc())

result = orders_df.withColumn("amount_dense_rank", dense_rank().over(w_country_amount))
```

Each country has its own rank sequence. Equal amounts share a rank; the next distinct amount gets the next consecutive rank.

### Example 2 — Top three distinct amount levels per country

```python
ranked = orders_df.withColumn("amount_level", dense_rank().over(w_country_amount))
top_three_levels = ranked.filter(col("amount_level") <= 3)
```

This retains rows from the top three distinct amount values per country. It may return more than three rows because multiple records can share each amount level.

### Example 3 — Dense-rank within category

```python
w_cat_amount = Window.partitionBy("category").orderBy(col("amount").desc())

result = orders_df.withColumn("category_amount_level", dense_rank().over(w_cat_amount))
```

The rank resets per category and counts distinct ordering values. This is useful for showing distinct value tiers within each category.

### Example 4 — Dense-rank customer totals

```python
customer_totals = orders_df.groupBy("customer_id").agg(
    sum("amount").alias("customer_total")
)

w_total = Window.orderBy(col("customer_total").desc())

result = customer_totals.withColumn("customer_total_level", dense_rank().over(w_total))
```

First calculate one total per customer, then dense-rank those totals. Customers with equal totals share the same rank. A global window may be costly when there are many customers.

### Example 5 — Dense-rank per country by total customer amount

```python
totals = orders_df.groupBy("country", "customer_id").agg(
    sum("amount").alias("customer_total")
)

w_country_total = Window.partitionBy("country").orderBy(col("customer_total").desc())

result = totals.withColumn("country_customer_level", dense_rank().over(w_country_total))
```

This creates customer totals by country and then ranks them within their country. The output grain remains one row per country-customer.

### Example 6 — Dense-rank quantity levels

```python
w_qty = Window.partitionBy("category").orderBy(col("quantity").desc())

result = orders_df.withColumn("quantity_level", dense_rank().over(w_qty))
```

Equal quantities share a rank. This is useful when the question is about distinct quantity levels, not a unique row sequence.

### Example 7 — Dense-rank by date recency

```python
w_dates = Window.partitionBy("customer_id").orderBy(col("event_ts").desc())

result = orders_df.withColumn("date_recency_level", dense_rank().over(w_dates))
```

If multiple records share a timestamp, they share a dense rank. This groups distinct timestamps into consecutive recency levels. If every event must have a unique position, use `row_number` instead.

### Example 8 — Dense-rank statuses after an aggregate

```python
status_totals = orders_df.groupBy("country", "status").agg(
    sum("amount").alias("status_amount")
)

w_status_total = Window.partitionBy("country").orderBy(col("status_amount").desc())

result = status_totals.withColumn("status_amount_level", dense_rank().over(w_status_total))
```

The first stage calculates amount by country and status. The second ranks statuses within each country by amount total. This is a good example of combining grouped aggregation and window ranking.

### Example 9 — Dense-rank in large data

```python
w_large_dense = Window.partitionBy("country").orderBy(col("amount").desc())

large_dense_ranked = large_orders_df.withColumn(
    "amount_level", dense_rank().over(w_large_dense)
)
```

This ranks distinct amount levels within each country across a large dataset. It may require sorting within each country. Inspect Spark UI and avoid collecting the full output.

### Example 10 — Compare all three ranking functions

```python
w_rank_compare = Window.partitionBy("country").orderBy(col("amount").desc())

result = orders_df.withColumn("row_num", row_number().over(w_rank_compare)) \
    .withColumn("rank_value", rank().over(w_rank_compare)) \
    .withColumn("dense_rank_value", dense_rank().over(w_rank_compare))
```

Use this to examine ties directly. The ranking functions differ only when the ordering values tie: `row_number` is unique, `rank` leaves gaps, and `dense_rank` uses consecutive distinct-value ranks.

---

# Part 8 — `lag()`: definition and 10 examples

## Definition and parameters

`lag(column, offset=1, default=None)` returns the value of a column from a preceding row in the ordered window. It is commonly used for comparing current versus previous events.

**Syntax**
```python
lag("amount", 1, None).over(window_spec)
```

- `column`: column whose prior value is returned.
- `offset`: number of rows backward; default is 1.
- `default`: value returned when the requested preceding row does not exist; default is `None`/null.
- `.over(window_spec)`: defines the sequence and partition.
- `lag` follows row order, not elapsed time. Offset 1 means previous row, not “one day earlier.”

### Example 1 — Previous order amount for each customer

```python
w_customer_time = Window.partitionBy("customer_id").orderBy("event_ts", "order_id")

result = orders_df.withColumn(
    "previous_amount", lag("amount", 1).over(w_customer_time)
)
```

For each customer, `lag` returns the amount from the previous ordered row. The first row for each customer has no previous row, so the result is null. This supports change analysis and customer journey features.

### Example 2 — Compare current amount with previous amount

```python
result = orders_df.withColumn(
    "previous_amount", lag("amount", 1).over(w_customer_time)
).withColumn(
    "amount_change", col("amount") - col("previous_amount")
)
```

The second expression subtracts the previous amount from the current amount. A positive value means the current amount is larger; a negative value means it is smaller. The first row's change is null because there is no previous amount.

### Example 3 — Calculate percentage change

```python
result = orders_df.withColumn(
    "previous_amount", lag("amount", 1).over(w_customer_time)
).withColumn(
    "amount_change_pct",
    (col("amount") - col("previous_amount")) / col("previous_amount") * 100
)
```

This calculates percentage change relative to the prior amount. If previous amount is null or zero, the result is null or undefined; guard against zero denominators in production. The metric compares adjacent rows, not necessarily adjacent days.

### Example 4 — Previous order status

```python
result = orders_df.withColumn(
    "previous_status", lag("status", 1).over(w_customer_time)
)
```

This returns the preceding status in each customer's ordered history. It can help detect transitions such as Pending to Delivered when the dataset contains status events. If the source has only one row per order and status is not a history of changes, it compares different orders' statuses instead.

### Example 5 — Previous event timestamp and time gap

```python
result = orders_df.withColumn(
    "previous_ts", lag("event_ts", 1).over(w_customer_time)
).withColumn(
    "hours_since_previous",
    (col("event_ts").cast("long") - col("previous_ts").cast("long")) / 3600
)
```

`lag` retrieves the previous timestamp. Casting timestamps to long obtains epoch seconds in Spark; subtracting and dividing by 3600 yields hours. The first row has no previous timestamp. For precise duration handling and timezone rules, verify timestamp semantics in the environment.

### Example 6 — Previous order date

```python
result = orders_df.withColumn(
    "previous_date", lag("event_date", 1).over(w_customer_time)
).withColumn(
    "days_since_previous_order",
    datediff(col("event_date"), col("previous_date"))
)
```

`lag` returns the previous date and `datediff` computes the number of calendar days between the current and previous dates. If a customer has multiple orders on the same date, the difference can be zero. This measures date difference, not exact elapsed 24-hour periods.

### Example 7 — Use a two-row offset

```python
result = orders_df.withColumn(
    "amount_two_orders_ago", lag("amount", 2).over(w_customer_time)
)
```

The offset `2` asks for the value two rows before the current row in the customer sequence. It does not mean two days before. The first two rows per customer have no row two positions behind them, so their result is null.

### Example 8 — Supply a default value

```python
result = orders_df.withColumn(
    "previous_amount_or_zero",
    lag("amount", 1, 0.0).over(w_customer_time)
)
```

The third argument `0.0` is returned when no previous row exists. This can simplify arithmetic, but it changes the meaning: zero is an assumption, not an observed previous amount. Use null when “no previous observation” must remain distinguishable from a real zero.

### Example 9 — Compare with previous event within category

```python
w_customer_category = Window.partitionBy(
    "customer_id", "category"
).orderBy("event_ts", "order_id")

result = orders_df.withColumn(
    "previous_category_amount", lag("amount", 1).over(w_customer_category)
)
```

The partition key includes both customer and category, so the previous value comes from the same customer's previous order in the same category. Other categories do not affect the sequence. This is useful for category-specific customer behaviour.

### Example 10 — Lag on a large dataset

```python
w_large_lag = Window.partitionBy("customer_id").orderBy("event_ts", "order_id")

large_with_previous = large_orders_df.withColumn(
    "previous_amount", lag("amount", 1).over(w_large_lag)
).withColumn(
    "amount_change", col("amount") - col("previous_amount")
)
```

This calculates adjacent-record amount changes across one million synthetic rows. Sorting/partitioning by customer and timestamp can be expensive. Inspect the plan and task distribution; do not collect the complete result.

**Lag pitfalls:** offset is a row offset, not a time interval; first rows usually return null; order must be deterministic; percentage calculations need zero/null handling.

---

# Part 9 — `lead()`: definition and 10 examples

## Definition and parameters

`lead(column, offset=1, default=None)` returns a value from a following row in the ordered window. It is useful for next-event analysis, gap detection, sequence validation, and identifying the next status or transaction.

- `column`: column whose future/next value is returned.
- `offset`: number of rows forward; default is 1.
- `default`: returned when the following row does not exist.
- `.over(window_spec)`: defines the partition and row ordering.
- `lead` follows row positions, not elapsed time. Offset 1 means next row.

### Example 1 — Next order amount

```python
result = orders_df.withColumn(
    "next_amount", lead("amount", 1).over(w_customer_time)
)
```

For each customer, `lead` returns the next ordered record's amount. The last row for each customer has no following row and returns null. This is useful for analysing what happens after an event.

### Example 2 — Difference between next and current amount

```python
result = orders_df.withColumn(
    "next_amount", lead("amount", 1).over(w_customer_time)
).withColumn(
    "next_amount_change", col("next_amount") - col("amount")
)
```

The calculation subtracts the current amount from the next amount. Positive means the next amount is higher. The last row's difference is null because there is no next amount.

### Example 3 — Next event timestamp and time until next event

```python
result = orders_df.withColumn(
    "next_ts", lead("event_ts", 1).over(w_customer_time)
).withColumn(
    "hours_until_next",
    (col("next_ts").cast("long") - col("event_ts").cast("long")) / 3600
)
```

`lead` retrieves the next timestamp. Subtracting the current epoch seconds from the next epoch seconds gives the elapsed time; dividing by 3600 expresses it in hours. The last event has no next timestamp. Verify timezone handling for production event-time calculations.

### Example 4 — Next status in customer sequence

```python
result = orders_df.withColumn(
    "next_status", lead("status", 1).over(w_customer_time)
)
```

This returns the status on the customer's next ordered record. It is meaningful for event histories; if rows represent unrelated orders with current status snapshots, it does not necessarily mean the current order's status transition.

### Example 5 — Next order date and days until next order

```python
result = orders_df.withColumn(
    "next_date", lead("event_date", 1).over(w_customer_time)
).withColumn(
    "days_until_next_order", datediff(col("next_date"), col("event_date"))
)
```

`datediff` computes calendar-day difference between current and next dates. Orders on the same date can produce zero. The last row per customer returns null because no next date exists.

### Example 6 — Look two events ahead

```python
result = orders_df.withColumn(
    "amount_two_orders_ahead", lead("amount", 2).over(w_customer_time)
)
```

Offset 2 looks two row positions forward. The last two rows per customer do not have two following rows, so they return null. This is useful for sequence patterns but not a substitute for a time-based join.

### Example 7 — Supply a default for missing next rows

```python
result = orders_df.withColumn(
    "next_amount_or_zero", lead("amount", 1, 0.0).over(w_customer_time)
)
```

The default `0.0` appears when no following row exists. Use it only if zero is the correct business interpretation; otherwise preserve null so that “no next event” is not confused with an event whose amount is zero.

### Example 8 — Next event within the same category

```python
w_customer_category = Window.partitionBy(
    "customer_id", "category"
).orderBy("event_ts", "order_id")

result = orders_df.withColumn(
    "next_same_category_amount", lead("amount", 1).over(w_customer_category)
)
```

The next value is selected within the same customer-category group. An order in another category does not become the next row for this calculation. This supports repeat-category purchase analysis.

### Example 9 — Detect a gap between events

```python
result = orders_df.withColumn(
    "next_ts", lead("event_ts", 1).over(w_customer_time)
).withColumn(
    "hours_to_next",
    (col("next_ts").cast("long") - col("event_ts").cast("long")) / 3600
).withColumn(
    "gap_over_48_hours",
    col("hours_to_next") > 48
)
```

The code derives the gap to the next event and flags gaps greater than 48 hours. The threshold is illustrative. The final row per customer has a null gap, so its boolean flag is null rather than false; decide how to handle end-of-sequence rows.

### Example 10 — Lead on a million-row dataset

```python
w_large_lead = Window.partitionBy("customer_id").orderBy("event_ts", "order_id")

large_with_next = large_orders_df.withColumn(
    "next_amount", lead("amount", 1).over(w_large_lead)
)
```

This adds each customer's next amount to a million-row dataset. The window needs rows to be ordered within each customer. Inspect the physical plan and Spark UI; do not collect the full output to the driver.

**Lead pitfalls:** offset is row-based, not time-based; last rows return null unless a default is specified; ordering must be stable; do not treat the next record as a future event outside the data available to the job.

---

# Part 10 — Value-window functions beyond `lag` and `lead`

## 1. What are value / analytic functions?

Value functions return a value from a selected row within the window/frame. `lag` and `lead` choose rows relative to the current row. `first_value`, `last_value`, and `nth_value` refer to positions within a window frame.

### Example 1 — First amount in each customer's history

```python
w_customer_full = Window.partitionBy("customer_id").orderBy(
    "event_ts", "order_id"
).rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

result = orders_df.withColumn(
    "first_customer_amount", first("amount", ignorenulls=True).over(w_customer_full)
)
```

This uses an explicit full-partition frame. `first(..., ignorenulls=True)` returns the first non-null amount in the ordered frame. The output keeps each row and attaches the same first amount for that customer.

### Example 2 — Last amount in the full customer history

```python
result = orders_df.withColumn(
    "last_customer_amount", last("amount", ignorenulls=True).over(w_customer_full)
)
```

The explicit frame runs from the first through the last row in the partition. This makes `last` refer to the last non-null amount in the complete ordered history. Without a full frame, `last` can mean the last value in a frame that ends at the current row.

### Example 3 — Second amount in the customer history

```python
result = orders_df.withColumn(
    "second_customer_amount", nth_value("amount", 2).over(w_customer_full)
)
```

`nth_value` uses a one-based position: 2 means the second value in the ordered frame. If the frame has fewer than two rows, the result is null. Verify function availability in the Spark version being used.

### Example 4 — First timestamp and last timestamp

```python
result = orders_df.withColumn(
    "first_ts", first("event_ts", ignorenulls=True).over(w_customer_full)
).withColumn(
    "last_ts", last("event_ts", ignorenulls=True).over(w_customer_full)
)
```

This attaches the earliest and latest timestamps in each customer's full history. `ignorenulls=True` skips null values for the value selection. These fields can support tenure or recency features.

### Example 5 — Compare current amount with first amount

```python
result = orders_df.withColumn(
    "first_amount", first("amount", ignorenulls=True).over(w_customer_full)
).withColumn(
    "difference_from_first", col("amount") - col("first_amount")
)
```

The first expression selects the customer's first non-null amount. The second compares each order to it. The result is row-level change from the baseline; it is not a time-normalized growth rate.

### Example 6 — Use a current-row cumulative frame for last value so far

```python
w_to_current = (
    Window.partitionBy("customer_id")
    .orderBy("event_ts", "order_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

result = orders_df.withColumn(
    "last_amount_seen_so_far", last("amount", ignorenulls=True).over(w_to_current)
)
```

The frame ends at the current row, so the result means the last non-null amount seen up through that row, not the customer's final amount in the entire dataset. The distinction is important when avoiding future information leakage.

### Example 7 — First event status for each customer

```python
result = orders_df.withColumn(
    "first_status", first("status", ignorenulls=True).over(w_customer_full)
)
```

This attaches the first non-null status in the customer's ordered records. In this sample, rows are order snapshots, not necessarily a status-change log, so interpret it as the first observed status among these records.

### Example 8 — Last category observed for each customer

```python
result = orders_df.withColumn(
    "last_category", last("category", ignorenulls=True).over(w_customer_full)
)
```

The full frame means the value comes from the customer's final ordered record. This can be used as a feature, but it leaks future information if applied to a historical prediction task. For as-of features, use only information available at the prediction time.

### Example 9 — Value functions on a large dataset

```python
w_large_full = Window.partitionBy("customer_id").orderBy(
    "event_ts", "order_id"
).rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

large_value_df = large_orders_df.withColumn(
    "first_amount", first("amount", ignorenulls=True).over(w_large_full)
).withColumn(
    "last_amount", last("amount", ignorenulls=True).over(w_large_full)
)
```

This attaches first and last values per customer to each row. It can require sorting and retaining window information. Test with a manageable volume and inspect execution before scaling further.

### Example 10 — Choose the right value function

```python
result = orders_df.withColumn(
    "previous_amount", lag("amount", 1).over(w_customer_time)
).withColumn(
    "next_amount", lead("amount", 1).over(w_customer_time)
).withColumn(
    "first_amount", first("amount", ignorenulls=True).over(w_customer_full)
).withColumn(
    "last_amount", last("amount", ignorenulls=True).over(w_customer_full)
)
```

The result contains four different concepts: preceding row, following row, first value in the full frame, and last value in the full frame. `lag`/`lead` are relative to the current row; first/last depend on the specified frame. For current Spark versions, `first` and `last` are commonly used as aggregate functions over a window; check your environment's support for `first_value`, `last_value`, and `nth_value` if you prefer those SQL-style names.

---

# Part 11 — Practical Data Engineering recipes

## Recipe 1 — Latest record per business key

```python
w_latest = Window.partitionBy("customer_id").orderBy(
    col("event_ts").desc(), col("order_id").desc()
)

latest_state_df = orders_df.withColumn(
    "rn", row_number().over(w_latest)
).filter(col("rn") == 1).drop("rn")
```

**Why it works:** each business key has its own window; descending time places the latest record first; `row_number == 1` retains one row. In production, define duplicate keys and tie-breaking rules explicitly.

## Recipe 2 — Top three records per country

```python
w_top = Window.partitionBy("country").orderBy(col("amount").desc(), col("order_id"))

top_three_df = orders_df.withColumn(
    "rn", row_number().over(w_top)
).filter(col("rn") <= 3)
```

This returns at most three rows per country. Use `rank` instead if ties at the boundary should all be retained.

## Recipe 3 — Customer running total and previous-event delta

```python
w_customer_running = (
    Window.partitionBy("customer_id")
    .orderBy("event_ts", "order_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

customer_timeline_df = orders_df.withColumn(
    "running_amount", sum("amount").over(w_customer_running)
).withColumn(
    "previous_amount", lag("amount", 1).over(w_customer_time)
).withColumn(
    "amount_delta", col("amount") - col("previous_amount")
)
```

This provides both cumulative amount and adjacent-order change. The running total includes the current row; for a feature that must not include the current order, use a frame ending at `Window.currentRow - 1` where supported by the chosen frame API, or compute the previous cumulative total with `lag` after defining a deterministic cumulative result.

## Recipe 4 — Compare daily amount with previous available day

First aggregate to one row per date:
```python
daily_df = orders_df.groupBy("event_date").agg(
    sum("amount").alias("daily_amount")
)

w_daily = Window.orderBy("event_date")

daily_change_df = daily_df.withColumn(
    "previous_available_day_amount", lag("daily_amount", 1).over(w_daily)
).withColumn(
    "daily_change", col("daily_amount") - col("previous_available_day_amount")
)
```

This compares each date with the previous date that exists in the aggregate. If a calendar date has no records, it will not appear; create a date spine first if you need gaps represented as zero or null.

## Recipe 5 — Identify large changes from the previous event

```python
change_df = orders_df.withColumn(
    "previous_amount", lag("amount", 1).over(w_customer_time)
).withColumn(
    "amount_delta", col("amount") - col("previous_amount")
).withColumn(
    "large_change_flag", abs(col("amount_delta")) >= 1000
)
```

This flags adjacent-order changes whose absolute value is at least 1000. The threshold is illustrative. Import `abs` from `pyspark.sql.functions` if using this recipe. A change flag is not proof of an anomaly; calibrate thresholds using business rules or historical distributions.

## Recipe 6 — Detect missing sequence time / long event gaps

```python
gap_df = orders_df.withColumn(
    "next_ts", lead("event_ts", 1).over(w_customer_time)
).withColumn(
    "gap_hours", (col("next_ts").cast("long") - col("event_ts").cast("long")) / 3600
).filter(col("gap_hours") > 48)
```

This finds records followed by a gap longer than 48 hours in the same customer's observed sequence. The final event has a null next timestamp and is filtered out by the comparison. If gaps are measured against expected event frequency, define the expected schedule.

## Recipe 7 — Ranking after aggregation

```python
category_totals = orders_df.groupBy("country", "category").agg(
    sum("amount").alias("category_amount")
)

w_category_rank = Window.partitionBy("country").orderBy(col("category_amount").desc())

ranked_categories = category_totals.withColumn(
    "category_rank", dense_rank().over(w_category_rank)
)
```

This first creates one row per country-category, then ranks categories by total amount within each country. It is more meaningful for category performance than ranking raw orders.

## Recipe 8 — Prevent future-data leakage

For an as-of feature, use only rows available at or before the current event. For example, a cumulative amount:
```python
w_as_of = (
    Window.partitionBy("customer_id")
    .orderBy("event_ts", "order_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

as_of_df = orders_df.withColumn("amount_as_of_event", sum("amount").over(w_as_of))
```

Do not attach a customer's full-history total to an old order if the feature is meant to represent what was known at that time. Full-partition first/last/total features can leak information from the future.

---

# Part 12 — Large-data performance and correctness

## 1. Window operations can be expensive

Ranking and sequence functions usually need rows to be partitioned and ordered. This can introduce shuffle and sort stages. A one-million-row example is useful, but runtime depends on:
- number of Spark partitions,
- number and size of groups,
- skewed keys,
- available executor memory/cores,
- cluster/storage configuration,
- sort keys and physical plan.

Inspect:
```python
large_orders_df.withColumn(
    "rn", row_number().over(w_large_sequence)
).explain("formatted")
```

Use Spark UI to inspect shuffle read/write, task duration, spill, and skew. Do not assume a particular execution plan without checking.

## 2. Avoid a global window unless you need it

```python
Window.orderBy(col("amount").desc())
```
This orders all rows in one logical window. It may be costly at scale. If the business question is “top N per country,” use `partitionBy("country")`.

## 3. Partition size and skew

A customer with millions of events can make one window partition much larger than others. That can create a long-running task. Measure group sizes and skew before changing partitioning. Repartitioning by the same key may help distribution in some workloads, but it does not eliminate the need to sort within each window.

## 4. Avoid unnecessary wide window columns

Each added window column increases output width. Calculate only metrics needed downstream, and reuse equivalent window definitions where practical.

## 5. Window order is not final display order

The `orderBy` inside a window defines the calculation sequence; it does not guarantee the final DataFrame display or output-file order. Use a final `.orderBy(...)` if a sorted presentation is required.

## 6. Nulls and defaults

- `lag`/`lead` return null if the offset row does not exist and no default is supplied.
- `lag`/`lead` can also return null if the target column on the referenced row is null.
- `first`/`last` null handling depends on `ignorenulls`.
- Aggregate windows inherit aggregate null behaviour.
- Replacing null with zero changes semantics; do so only when zero is the intended value.

---

# Part 13 — Common mistakes checklist

1. Confusing `groupBy` aggregation with window aggregation.
2. Forgetting `.over(window_spec)`.
3. Using `row_number` when tied rows should share a rank.
4. Using `rank` when exactly N rows are required.
5. Using `dense_rank` while expecting rank gaps.
6. Forgetting `orderBy` for sequence-sensitive operations.
7. Using an unstable order when ties exist.
8. Assuming `lag(..., 1)` means one day earlier.
9. Assuming `lead` returns a future record outside the current dataset.
10. Using full-history metrics in historical machine-learning features and leaking future data.
11. Forgetting to specify a full frame when the intent is the first/last value in the entire partition.
12. Calling `collect()` on a large windowed DataFrame.
13. Assuming the window `orderBy` also sorts final output.
14. Treating synthetic values as actual business evidence.

---

# Part 14 — Quick reference

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, row_number, rank, dense_rank, lag, lead,
    sum, avg, count, min, max
)

# Customer sequence
w = Window.partitionBy("customer_id").orderBy("event_ts", "order_id")

# Row sequence
df.withColumn("rn", row_number().over(w))

# Rank with gaps after ties
df.withColumn("r", rank().over(w))

# Rank without gaps after ties
df.withColumn("dr", dense_rank().over(w))

# Previous value
df.withColumn("previous_amount", lag("amount", 1).over(w))

# Next value
df.withColumn("next_amount", lead("amount", 1).over(w))

# Full-partition total attached to every row
w_customer = Window.partitionBy("customer_id")
df.withColumn("customer_total", sum("amount").over(w_customer))

# Running total
w_running = (
    Window.partitionBy("customer_id")
    .orderBy("event_ts", "order_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
df.withColumn("running_amount", sum("amount").over(w_running))

# Rolling average of current and previous two rows
w_rolling = (
    Window.partitionBy("customer_id")
    .orderBy("event_ts", "order_id")
    .rowsBetween(-2, Window.currentRow)
)
df.withColumn("rolling_avg", avg("amount").over(w_rolling))
```

## Final mental model

- **Aggregate window:** calculate totals/counts/averages/extremes while preserving the input rows.
- **Ranking window:** give rows an ordered position or rank within a group.
- **Value window:** retrieve previous, next, first, last, or nth values from a specified ordered window/frame.
- **`row_number()`** always gives a unique sequential position.
- **`rank()`** preserves ties and leaves gaps.
- **`dense_rank()`** preserves ties without gaps.
- **`lag()`** looks backward by row offset.
- **`lead()`** looks forward by row offset.
- For Data Engineering, use windows for latest-record selection, deduplication, running metrics, change detection, top-N analysis, and customer/event features.
- For scale, choose partition keys intentionally, use deterministic ordering, inspect execution plans, and avoid collecting large outputs.
